In [6]:
import pandas as pd
import numpy as np

df=pd.read_csv(r"../resource/IMDB Dataset.csv")

In [7]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [8]:
df.shape

(50000, 2)

In [9]:
type(df)

pandas.DataFrame

In [10]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [11]:
df.replace({"sentiment":{"positive":1, "negative":0}}, inplace=True)

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1
...,...,...
49995,I thought this movie did a down right good job...,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0
49997,I am a Catholic taught in parochial elementary...,0
49998,I'm going to have to disagree with the previou...,0


In [12]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [13]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [79]:
x_train, x_test, y_train, y_test=train_test_split(df['review'], df['sentiment'], test_size=0.2, random_state=2)

In [80]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(40000,)
(10000,)
(40000,)
(10000,)


In [67]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(x_train)
x_train_tr = pad_sequences(tokenizer.texts_to_sequences(x_train), maxlen=200)

In [68]:
print(x_train_tr.shape)

(40000, 200)


In [69]:
x_train_tr

array([[3474,   13,  847, ...,   78,  547,  166],
       [   0,    0,    0, ...,  105, 3444,  176],
       [ 133,    6,  429, ...,  143,  155, 1198],
       ...,
       [ 195,  117,   32, ...,   27,    4,   91],
       [   0,    0,    0, ...,   19,   30,  125],
       [  38,   88, 2252, ...,   23,   30,    9]],
      shape=(40000, 200), dtype=int32)

In [60]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation="sigmoid"))

In [61]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [70]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [94]:
y_train = y_train.astype('float32')
model.fit(x_train_tr, y_train, epochs=5, batch_size=4, validation_split=0.2)

Epoch 1/5
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 1120s 140ms/step - accuracy: 0.8136 - loss: 0.4032 - val_accuracy: 0.8844 - val_loss: 0.2900
Epoch 2/5
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 3439s 430ms/step - accuracy: 0.9072 - loss: 0.2283 - val_accuracy: 0.9009 - val_loss: 0.2526
Epoch 3/5
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 1185s 148ms/step - accuracy: 0.9336 - loss: 0.1713 - val_accuracy: 0.8969 - val_loss: 0.2771
Epoch 4/5
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 2055s 257ms/step - accuracy: 0.9519 - loss: 0.1266 - val_accuracy: 0.8891 - val_loss: 0.3023
Epoch 5/5
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 4668s 584ms/step - accuracy: 0.9678 - loss: 0.0884 - val_accuracy: 0.8891 - val_loss: 0.3313


In [96]:
model.save("model_2.keras")

In [105]:
x_test_tr = pad_sequences(tokenizer.texts_to_sequences(x_test), maxlen=200)
x_test_tr = np.asarray(x_test_tr, dtype='int32')
y_test = np.asarray(y_test, dtype='float32')

In [106]:
loss, accuracy = model.evaluate(x_test_tr, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.8893 - loss: 0.3332


In [107]:
loss

0.33317288756370544

In [108]:
accuracy

0.8892999887466431